## Прогноз объема BTC и ETH

Прогнозирем логарифм объема наследующий час и логарифм суммарного объема следующих 24 часов.
Модель - AR(1)

In [ ]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.stats import norm
from sklearn.linear_model import Lasso
from statsmodels.api import OLS
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.multitest  import multipletests
from IPython.display import display


RUN_BOOTSTRAP = True
BOOT_REPS = 4999
SEED = 20261026
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
plt.rcParams.update({
    'figure.figsize': (10, 3.5),
    'axes.grid': True,
    'grid.alpha': .18,
    'axes.spines.top': False,
    'axes.spines.right': False
})

In [ ]:
raw = pd.read_csv("../data/ohlcv_1h.csv", parse_dates=['time_utc'])
raw['time_utc'] = pd.to_datetime(raw.time_utc, utc=True)
assert not raw.duplicated(['symbol', 'time_utc']).any()
assert raw[['open', 'high', 'low', 'close', 'volume']].notna().all().all()
assert (raw[['open', 'high', 'low', 'close', 'volume']] > 0).all().all()
assert (raw.high >= raw[['open','close','low']].max(axis=1)).all()
assert (raw.low <= raw[['open','close','high']].min(axis=1)).all()

P = {}
for field in ['open','high','low','close','volume']:
    z = raw.pivot(index='time_utc', columns='symbol', values=field).sort_index()[SYMBOLS]
    assert z.index.equals(pd.date_range(z.index[0], z.index[-1], freq='h'))
    assert not z.isna().any().any()
    z.index += pd.Timedelta(hours=1)
    P[field] = z.asfreq('h')

H = (100 * np.log(P['close']).diff()).dropna()  #часовые лог-доходности

In [ ]:
display(H)

In [ ]:
H.describe()

## Подготовка выборки

Берем полные сутки по 24 доходности. Из 401 суток после разбиения:

Train: 233 

Test: 112

Validation: 56

In [ ]:
counts = H.resample('D', closed='right', label='right').count().min(axis=1)
days = counts[counts == 24].index
assert len(days) == 401
TRAIN_DAYS, VALID_DAYS, TEST_DAYS = 233, 56, 112
vs = H.index.get_loc(days[TRAIN_DAYS-1]) + 1
ts = H.index.get_loc(days[TRAIN_DAYS+VALID_DAYS-1]) + 1
n = len(H)
assert n - ts == TEST_DAYS * 24

print(f'Validation: {H.index[vs]} - {H.index[ts-1]}')
print(f'Test: {H.index[ts]} - {H.index[-1]}')
print(f'Номера разбиения: {dict(vs=vs, ts=ts, n=n)}')

## Перобразование объема

Цель: $Log(Volume_{t+1})$

Для суммарного объема берем: $Log(volume_{t+1} + ... + volume_{t+24})$

+Построение календарного профиля

In [ ]:
def future_point(y, h=1):
    out = np.full(len(y), np.nan)
    out[:-h] = np.asarray(y)[h:]
    return out

def future_sum(y, h):
    a = np.asarray(y, float)
    out = np.full(len(a), np.nan)
    cumulative = np.r_[0., np.cumsum(a)]
    out[:-h] = cumulative[h+1:] - cumulative[1:len(a)-h+1]
    return out

def make_frame(symbol):
    y = np.log(P['volume'][symbol]).reindex(H.index)
    z = pd.DataFrame({
        'v1': y, 'v2': y.shift(1), 'v24': y.shift(23), 'v168': y.shift(167),
        'mean24': y.rolling(24).mean(), 'mean168': y.rolling(168).mean(),
        'risk24': np.log((H[symbol]**2).rolling(24).sum()),
        'range1': np.log((100*np.log(P['high'][symbol]/P['low'][symbol]))**2).reindex(H.index),
        'return1': H[symbol]
    }, index=H.index)
    hour = np.asarray((H.index - pd.Timestamp('1970-01-01', tz='UTC')) / pd.Timedelta(hours=1)) + 1
    for period in [24, 168]:
        for k in [1, 2, 3]:
            z[f'sin_{period}_{k}'] = np.sin(2*np.pi*k*hour/period)
            z[f'cos_{period}_{k}'] = np.cos(2*np.pi*k*hour/period)
    return y, z

series = {s:make_frame(s) for s in SYMBOLS}
targets = {(s, h): future_point(series[s][0]) if h==1
           else np.log(future_sum(np.exp(series[s][0]), 24))
           for s in SYMBOLS for h in [1,24]}
for s in SYMBOLS:
    assert np.allclose(targets[s,1][:-1], series[s][0].to_numpy()[1:])
    assert np.isfinite(targets[s,24][ts-1:n-24]).all()

fig, ax = plt.subplots()
for symbol in SYMBOLS:
    y = np.exp(series[symbol][0].iloc[:vs])
    normal = y / y.mean()
    ax.plot(normal.groupby(normal.index.hour).mean(), marker='.', label=symbol)
ax.set(xlabel='Час закрытия свечи', ylabel='Средний объём',
       title='Внутрисуточный профиль')
ax.legend()
plt.show()

## AR(1), сезонные и расширенные модели

Оцениваем: AR(1), AR-seasonal, Calendar-ARX, Ridge, Lasso, PCR(3)
Дополнительно: Persistence, Seasonal24/168, ARIMAX-calendar

Стандартизируем внутри каждого тренировочного окна, регуляризация без штрафа наконстанту, PCA только по обучающим признакам

In [ ]:
def make_specs(columns):
    specs = [
        {'name':'AR1','kind':'OLS','cols':['v1']},
        {'name':'AR-seasonal','kind':'OLS',
         'cols':['v1','v2','v24','v168','mean24','mean168']},
        {'name':'Calendar-ARX','kind':'OLS','cols':list(columns)}
    ]
    for alpha in [.01, .1, 1, 10]:
        specs.append({'name':f'Ridge({alpha:g})','kind':'Ridge', 'cols':list(columns), 'alpha':alpha})
    for alpha in [.001, .01, .1]:
        specs.append({'name':f'Lasso({alpha:g})','kind':'Lasso', 'cols':list(columns), 'alpha':alpha})
    specs.append({'name':'PCR(3)','kind':'PCR','cols':list(columns), 'n':3})
    return specs

def linear_predict(X, y, x, spec):
    X, y, x = np.asarray(X,float), np.asarray(y,float), np.atleast_2d(x).astype(float)
    mask = np.isfinite(y) & np.isfinite(X).all(axis=1)
    X, y = X[mask], y[mask]
    assert len(y) >= 50 and np.isfinite(x).all()
    mu, sd = X.mean(0), np.maximum(X.std(0), 1e-9)
    z, xx = (X-mu)/sd, (x-mu)/sd
    ym, ys = y.mean(), max(y.std(), 1e-9)
    v = (y-ym)/ys
    A, xx1 = np.c_[np.ones(len(z)),z], np.c_[np.ones(len(xx)),xx]
    if spec['kind']=='OLS':
        out = xx1 @ np.linalg.lstsq(A, v, rcond=None)[0]
    elif spec['kind']=='Ridge':
        penalty = np.diag(np.r_[0., np.repeat(spec['alpha']*len(v), z.shape[1])])
        out = xx1 @ np.linalg.solve(A.T@A + penalty, A.T@v)
    elif spec['kind']=='Lasso':
        fit = Lasso(alpha=spec['alpha'], max_iter=100000, tol=1e-6).fit(z,v)
        out = fit.predict(xx)
    else:
        own, exo = z[:,:3], z[:,3:]
        ownx, exox = xx[:,:3], xx[:,3:]
        _,_,vt = np.linalg.svd(exo, full_matrices=False)
        load = vt[:spec['n']].T
        W = np.c_[np.ones(len(z)),own,exo@load]
        wx = np.c_[np.ones(len(xx)),ownx,exox@load]
        out = wx @ np.linalg.lstsq(W,v,rcond=None)[0]
    return ym+ys*np.asarray(out)

specs = {s:make_specs(series[s][1].columns) for s in SYMBOLS}
display(pd.DataFrame([{'Модель':sp['name'], 'Метод':sp['kind'],
                       'Число признаков':len(sp['cols'])} for sp in specs['BTCUSDT']]))

## Подбор параметров train

В каждом CV окне ежедневная переоценка по 2880 ч

Для Ridge и Lasso сохраняем лучший штраф по среднему MSE

Финальная процедура для каждой задачиопределяется минимумом MSE на валидационной выборке

In [ ]:
def cv_search(symbol, h):
    y, F = series[symbol]
    target = targets[symbol, h]
    candidates = specs[symbol]
    xx = {sp['name']: F[sp['cols']].to_numpy() for sp in candidates}
    errors = {sp['name']: [] for sp in candidates}
    for j in range(3):
        start = vs - 1512 + j * 504
        stop = start + 504
        for b in range(start, stop, 24):
            origins = np.arange(b, min(b + 24, stop - h + 1)) - 1
            if len(origins) == 0:
                continue
            right = b - h
            for sp in candidates:
                X = xx[sp['name']]
                pred = linear_predict(X[right - 2880:right], target[right - 2880:right], X[origins], sp)
                errors[sp['name']].extend((target[origins] - pred) ** 2)
    retained = [sp for sp in candidates if sp['kind'] in ('OLS', 'PCR')]
    for kind in ('Ridge', 'Lasso'):
        pool = [sp for sp in candidates if sp['kind'] == kind]
        retained.append(min(pool, key=lambda sp: np.mean(errors[sp['name']])))
    names = {sp['name'] for sp in retained}
    table = pd.DataFrame([{
        'series': symbol,
        'h': h,
        'model': sp['name'],
        'N_CV': len(errors[sp['name']]),
        'loss_CV': np.mean(errors[sp['name']]),
        'retained': sp['name'] in names
    } for sp in candidates])
    return retained, table


fitted_specs = {}
cv_results = []
for symbol in SYMBOLS:
    for horizon in [1, 24]:
        chosen_specs, report = cv_search(symbol, horizon)
        fitted_specs[symbol, horizon] = chosen_specs
        cv_results.append(report)
        print('CV', symbol, horizon, [sp['name'] for sp in chosen_specs])

cv_table = pd.concat(cv_results, ignore_index=True)
display(cv_table[cv_table.retained][['series', 'h', 'model', 'loss_CV']].round(4))

## Прямые rolling-прогнозы + ARIMAX-calendar

ARIMAX-calendar использует ARMA(1,1) с известными будущими Fourier-фазами: параметры переоцениваются раз в 168 часов, между подгонками обновляется состояние. Для горизонта 24 прогноз агрегата logsumexp 24 почасовых лог-прогнозов.

In [ ]:
WINDOW = 2880


def fourier(index):
    hours = np.asarray((index - pd.Timestamp('1970-01-01', tz='UTC')) / pd.Timedelta(hours=1))
    return np.column_stack([
        fn(2 * np.pi * k * hours / period)
        for period in [24, 168]
        for k in [1, 2, 3]
        for fn in [np.sin, np.cos]
    ])


def refit_arimax(symbol):
    y = series[symbol][0]
    paths = {1: np.full(n, np.nan), 24: np.full(n, np.nan)}
    log = []
    for b in range(vs, n + 1, 168):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fit = ARIMA(
                y.iloc[b - WINDOW:b].to_numpy(),
                exog=fourier(y.index[b - WINDOW:b]),
                order=(1, 0, 1),
                trend='c'
            ).fit(method_kwargs={'maxiter': 500})
        converged = bool(fit.mle_retvals.get('converged', True))
        log.append({'series': symbol, 'refit_origin': H.index[b - 1], 'converged': converged})
        stop = min(b + 167, n)
        for origin in range(b - 1, stop):
            if converged:
                future = pd.date_range(H.index[origin] + pd.Timedelta(hours=1), periods=24, freq='h')
                path = np.asarray(fit.forecast(24, exog=fourier(future)))
            else:
                path = y.iloc[origin - 23:origin + 1].to_numpy()
            paths[1][origin] = path[0]
            paths[24][origin] = logsumexp(path)
            if converged and origin + 1 < stop:
                fit = fit.extend([y.iloc[origin + 1]], exog=fourier(y.index[[origin + 1]]))
    return paths, pd.DataFrame(log)


def baseline_paths(symbol, h):
    y = series[symbol][0]
    if h == 1:
        return {
            'Persistence': y.to_numpy(),
            'Seasonal24': y.shift(23).to_numpy(),
            'Seasonal168': y.shift(167).to_numpy()
        }
    past_day = np.exp(y).rolling(24).sum()
    return {
        'Persistence': np.log(past_day).to_numpy(),
        'Seasonal24': np.log(past_day).to_numpy(),
        'Seasonal168': np.log(past_day.shift(144)).to_numpy()
    }


def refit_direct(symbol, h, candidates):
    F = series[symbol][1]
    target = targets[symbol, h]
    paths = baseline_paths(symbol, h)
    features = {sp['name']: F[sp['cols']].to_numpy() for sp in candidates}
    for sp in candidates:
        paths[sp['name']] = np.full(n, np.nan)
    for b in range(vs, n + 1, 24):
        origins = np.arange(b, min(b + 24, n + 1)) - 1
        right = b - h
        for sp in candidates:
            X = features[sp['name']]
            paths[sp['name']][origins] = linear_predict(
                X[right - WINDOW:right],
                target[right - WINDOW:right],
                X[origins],
                sp
            )
    return paths


def to_long(symbol, h, paths):
    target = targets[symbol, h]
    parts = []
    for split, idx in [('validation', np.arange(vs - 1, ts - h)), ('test', np.arange(ts - 1, n - h))]:
        for model, forecast in paths.items():
            parts.append(pd.DataFrame({
                'series': symbol,
                'h': h,
                'split': split,
                'model': model,
                'origin': H.index[idx],
                'actual': target[idx],
                'forecast': forecast[idx]
            }))
    return pd.concat(parts, ignore_index=True)


panels = []
last_forecasts = {}
arimax_logs = []
for symbol in SYMBOLS:
    arimax, log = refit_arimax(symbol)
    arimax_logs.append(log)
    for h in [1, 24]:
        paths = refit_direct(symbol, h, fitted_specs[symbol, h])
        paths['ARIMAX-calendar'] = arimax[h]
        last_forecasts[symbol, h] = {name: path[-1] for name, path in paths.items()}
        panels.append(to_long(symbol, h, paths))

panel = pd.concat(panels, ignore_index=True)
assert panel[['actual', 'forecast']].notna().all().all()
display(pd.concat(arimax_logs).groupby('series').converged.mean())

## Выбор по валидационной выбьорке и оценка

In [ ]:
panel['sq'] = (panel.actual - panel.forecast) ** 2
panel['abs'] = (panel.actual - panel.forecast).abs()

metrics = (
    panel.groupby(['series', 'h', 'split', 'model'])
    .agg(N=('sq', 'size'), MSE=('sq', 'mean'), MAE=('abs', 'mean'))
    .reset_index()
)
metrics['RMSE'] = np.sqrt(metrics.MSE)

baselines = ['AR1', 'Persistence', 'Seasonal168']
valid = metrics[(metrics.split == 'validation') & ~metrics.model.isin(baselines)]
best = valid.loc[valid.groupby(['series', 'h']).MSE.idxmin()]
chosen = {(r.series, r.h): r.model for r in best.itertuples()}
display(best[['series', 'h', 'model', 'MSE']].round(5))

test = metrics[metrics.split == 'test'].set_index(['series', 'h', 'model'])
rows = []
for (symbol, h), model in chosen.items():
    a = test.loc[(symbol, h, model)]
    b = test.loc[(symbol, h, 'AR1')]
    rows.append({
        'Монета': symbol,
        'h': h,
        'Модель': model,
        'N': int(a.N),
        'MSE модели': a.MSE,
        'MSE AR(1)': b.MSE,
        'Снижение MSE, %': 100 * (1 - a.MSE / b.MSE),
        'RMSE / AR(1)': a.RMSE / b.RMSE
    })
summary = pd.DataFrame(rows)
display(summary.round(4))

## Сравнение с AR(1)

In [ ]:
test_all = metrics[metrics.split == 'test'].copy()
base = test_all.loc[test_all.model == 'AR1', ['series', 'h', 'RMSE']].rename(columns={'RMSE': 'base'})
test_all = test_all.merge(base, on=['series', 'h'])
test_all['ratio'] = test_all.RMSE / test_all.base
test_all['task'] = test_all.series.str[:3] + ', h=' + test_all.h.astype(str)
ratios = test_all.pivot(index='model', columns='task', values='ratio')
display(ratios.round(3))

## Парные ошибки HAC и bootstrap

In [ ]:
book = {
    key: group.sort_values('origin').set_index('origin')
    for key, group in panel.groupby(['series', 'h', 'model', 'split'])
}
p_cols = ['pHAC1', 'pHAC2', 'pBoot1', 'pBoot2']


def loss_difference(symbol, h, model, benchmark):
    a = book[symbol, h, model, 'test']
    b = book[symbol, h, benchmark, 'test']
    v = book[symbol, h, benchmark, 'validation']
    scale = np.mean((v.actual - v.forecast) ** 2)
    loss_a = (a.actual - a.forecast) ** 2
    loss_b = (b.actual - b.forecast) ** 2
    return (loss_b - loss_a) / scale, 100 * (1 - loss_a.mean() / loss_b.mean())


def bootstrap_means(d, block, rng):
    size = len(d)
    blocks = int(np.ceil(size / block))
    offsets = np.arange(block)
    means = np.empty(BOOT_REPS)
    for start in range(0, BOOT_REPS, 250):
        count = min(250, BOOT_REPS - start)
        starts = rng.integers(0, size, size=(count, blocks))
        idx = ((starts[:, :, None] + offsets) % size).reshape(count, -1)[:, :size]
        means[start:start + count] = d[idx].mean(axis=1)
    return means


def inference(d, seed):
    d = np.asarray(d, float)
    mean = d.mean()
    half = len(d) // 2
    out = {
        'N': len(d),
        'mean_gain': mean,
        'first_half': d[:half].mean(),
        'second_half': d[half:].mean()
    }
    for j, block in enumerate([168, 336], 1):
        fit = OLS(d, np.ones((len(d), 1))).fit(
            cov_type='HAC',
            cov_kwds={'maxlags': block, 'use_correction': True}
        )
        out[f'pHAC{j}'] = norm.sf(mean / float(fit.bse[0]))
        means = bootstrap_means(d, block, np.random.default_rng(seed + block))
        out[f'pBoot{j}'] = (1 + np.sum(means - mean >= mean)) / (BOOT_REPS + 1)
    return out


def with_holm(table):
    table = table.copy()
    for col in p_cols:
        table[col + '_holm'] = multipletests(table[col], method='holm')[1]
    return table

## Гипотеза 1 (выбранные модели против AR(1))

In [ ]:
rows = []
gains = {}
for i, ((symbol, h), model) in enumerate(chosen.items()):
    d, pct = loss_difference(symbol, h, model, 'AR1')
    gains[f'{symbol[:3]}_{h}'] = d
    rows.append({
        'series': symbol,
        'h': h,
        'model': model,
        'gain_pct': pct,
        **inference(d, SEED + 100 * i)
    })

h1_table = with_holm(pd.DataFrame(rows))
holm_cols = [col + '_holm' for col in p_cols]
display(h1_table[['series', 'h', 'model', 'gain_pct', 'first_half', 'second_half'] + holm_cols].round(5))

pooled = pd.concat(gains, axis=1, join='inner').dropna()
pooled_mean = pooled.mean(axis=1)
pooled_stats = inference(pooled_mean, SEED + 4000)
display(pd.DataFrame([pooled_stats]).round(6))

## Гипотезы 2 и 3(фиксированные сравнения)

In [ ]:
pairs = [
    ('AR-seasonal', 'AR1'),
    ('Calendar-ARX', 'AR-seasonal'),
    ('ARIMAX-calendar', 'AR1')
]
rows = []
for symbol in SYMBOLS:
    for h in [1, 24]:
        for model, benchmark in pairs:
            d, pct = loss_difference(symbol, h, model, benchmark)
            rows.append({
                'series': symbol,
                'h': h,
                'model': model,
                'benchmark': benchmark,
                'gain_pct': pct,
                **inference(d, SEED + 7000 + 20 * len(rows))
            })

h23_table = with_holm(pd.DataFrame(rows))
display(h23_table[['series', 'h', 'model', 'benchmark', 'gain_pct'] + holm_cols].round(4))

## Устойчивость и MCS

In [ ]:
from arch.bootstrap import MCS

rows = []
for (symbol, h), model in chosen.items():
    d, _ = loss_difference(symbol, h, model, 'AR1')
    rows.append({
        'series': symbol,
        'h': h,
        'N_non_overlapping': len(d.iloc[::h]),
        'gain_non_overlapping': d.iloc[::h].mean()
    })
display(pd.DataFrame(rows).round(4))

forecasts = (
    panel[(panel.series == 'BTCUSDT') & (panel.h == 1) & (panel.split == 'test')]
    .pivot(index='origin', columns='model', values='forecast')
)
actual = book['BTCUSDT', 1, 'AR1', 'test'].actual.reindex(forecasts.index)
losses = forecasts.sub(actual, axis=0) ** 2
losses = losses.loc[:, ~losses.T.duplicated()]

mcs = MCS(losses, size=0.1, reps=1999, block_size=168, method='R', bootstrap='circular', seed=SEED + 4)
mcs.compute()
mcs_table = mcs.pvalues.copy()
mcs_table['included'] = mcs_table.index.isin(mcs.included)
display(mcs_table.sort_values('Pvalue', ascending=False).round(4))

## Прогнозы, временная динамика выигрышка и календарные эффекты

In [ ]:
window = book['BTCUSDT', 1, 'AR1', 'test'].iloc[:14 * 24]
fig, ax = plt.subplots()
ax.plot(window.index, window.actual, lw=.75, color='black', label='Факт')
for model, style in [('AR1', '--'), ('Calendar-ARX', '-')]:
    forecast = book['BTCUSDT', 1, model, 'test'].forecast.reindex(window.index)
    ax.plot(forecast.index, forecast, lw=1, ls=style, label=model)
ax.set(
    xlabel='Origin (UTC)',
    ylabel='Логарифм часового объёма',
    title='BTC, h=1: первые две недели test'
)
ax.legend()
plt.show()

fig, ax = plt.subplots()
for col in pooled.columns:
    ax.plot(pooled.index, pooled[col].cumsum(), alpha=.35, lw=.8, label=col)
ax.plot(pooled.index, pooled_mean.cumsum(), color='black', lw=2, label='Среднее')
ax.axhline(0, color='gray', ls=':')
ax.set(
    xlabel='Origin (UTC)',
    ylabel='Накопленный нормированный выигрыш',
    title='MSE AR(1) минус MSE выбранной модели'
)
ax.legend(ncol=3, fontsize=8)
plt.show()

## Прогноз от последнего исторического среза

In [ ]:
last_origin = H.index[-1]
future = pd.DataFrame([
    {
        'series': symbol,
        'h': h,
        'origin': last_origin,
        'target_end': last_origin + pd.Timedelta(hours=h),
        'model': model,
        'forecast': last_forecasts[symbol, h][model]
    }
    for (symbol, h), model in chosen.items()
])
display(future.round(5))